# 05 — Génération de la soumission finale (Phase 6)

Correspond à la Phase 6 de `docs/WORKFLOW.md`. Charge l'adaptateur LoRA final, génère les réponses pour toutes les questions de `Test.csv`, et construit le fichier de soumission au format attendu par Zindi.

## 1 — Environnement

In [ ]:
!pip install -q -U transformers accelerate peft bitsandbytes datasets pandas

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/gemmafro-e2b'
ADAPTER_DIR = f'{PROJECT_DIR}/checkpoints/gemma-4-e2b-lora-final'
SUBMISSIONS_DIR = f'{PROJECT_DIR}/submissions'
os.makedirs(SUBMISSIONS_DIR, exist_ok=True)
assert os.path.isdir(ADAPTER_DIR), "Adaptateur introuvable — exécuter 03_finetune.ipynb jusqu'au bout d'abord."

In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))

## 2 — Charger `Test.csv`

In [ ]:
import pandas as pd

REPO_DIR = '/content/gemmafro-e2b'
if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/andilMc/gemmafro-e2b.git {REPO_DIR}

test = pd.read_csv(f'{REPO_DIR}/data/Test.csv')
sample_submission = pd.read_csv(f'{REPO_DIR}/data/SampleSubmission.csv')
print(test.shape)
test.head(3)

## 3 — Charger le modèle de base + l'adaptateur LoRA
Identique à `04_evaluate.ipynb` : mêmes réglages 4-bit/bf16, même déballage `Gemma4ClippableLinear` avant de charger l'adaptateur.

In [ ]:
import gc
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

MODEL_NAME = "google/gemma-4-E2B-it"
MAX_SEQ_LENGTH = 512

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'

gc.collect()
torch.cuda.empty_cache()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map={"": 0},
)
base_model.config.pad_token_id = tokenizer.pad_token_id
base_model.config.use_cache = True

def unwrap_clippable_linears(model):
    count = 0
    for module in model.modules():
        for child_name, child in list(module.named_children()):
            if child.__class__.__name__ == "Gemma4ClippableLinear":
                setattr(module, child_name, child.linear)
                count += 1
    print(f'{count} couches Gemma4ClippableLinear déballées vers Linear4bit')
    return model

base_model = unwrap_clippable_linears(base_model)

model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.eval()

gc.collect()
torch.cuda.empty_cache()
print(f"Mémoire GPU allouée : {torch.cuda.memory_allocated() / 1e9:.2f} Go")

## 4 — Construire les prompts (même format qu'en Phase 2/3)

In [ ]:
SUBSET_TO_LANGUAGE = {
    'Eng': 'English',
    'Aka': 'Akan',
    'Lug': 'Luganda',
    'Swa': 'Swahili',
    'Amh': 'Amharic',
}

def subset_to_language_name(subset_code: str) -> str:
    if not subset_code or not isinstance(subset_code, str):
        return 'English'
    return SUBSET_TO_LANGUAGE.get(subset_code.split('_')[0], subset_code)

def build_prompt(question: str, language: str) -> str:
    return (
        f"Réponds à la question de santé suivante en {language}, "
        f"de façon claire et médicalement fiable.\n\nQuestion : {question}"
    )

test['prompt_text'] = test.apply(
    lambda row: tokenizer.apply_chat_template(
        [{"role": "user", "content": build_prompt(row['input'], subset_to_language_name(row['subset']))}],
        tokenize=False, add_generation_prompt=True,
    ),
    axis=1,
)

## 5 — Générer les réponses pour tout `Test.csv`

In [ ]:
import re

@torch.no_grad()
def generate_answers(prompts, batch_size=8, max_new_tokens=400):
    answers = []
    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i + batch_size]
        inputs = tokenizer(
            batch, return_tensors='pt', padding=True, truncation=True,
            max_length=MAX_SEQ_LENGTH, add_special_tokens=False,
        ).to(model.device)
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
        new_tokens = out[:, inputs['input_ids'].shape[1]:]
        decoded = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)
        answers.extend(d.strip() for d in decoded)
        if (i // batch_size) % 10 == 0:
            print(f'{min(i + batch_size, len(prompts))}/{len(prompts)}')
    return answers

print(f'Génération pour {len(test)} questions de test...')
test_predictions = generate_answers(test['prompt_text'].tolist())
print('\n✅ Génération terminée')

## 6 — Construire et valider le fichier de soumission

In [ ]:
def make_submission(ids, predictions, output_path, reference_ids):
    # Belt-and-suspenders : retire d'éventuels tokens spéciaux résiduels.
    clean_preds = [re.sub(r'<[^>]+>', '', str(p)).strip() for p in predictions]

    sub = pd.DataFrame({
        'ID': ids,
        'TargetRLF1': clean_preds,
        'TargetR1F1': clean_preds,
        'TargetLLM': clean_preds,
    })[['ID', 'TargetRLF1', 'TargetR1F1', 'TargetLLM']]

    # Vérifications avant sauvegarde.
    assert set(sub['ID']) == set(reference_ids), "Les ID ne correspondent pas exactement à Test.csv"
    assert not sub['ID'].duplicated().any(), "IDs dupliqués dans la soumission"
    assert not sub[['TargetRLF1', 'TargetR1F1', 'TargetLLM']].isna().any().any(), "Valeurs manquantes"
    assert (sub['TargetRLF1'].str.len() > 0).all(), "Réponses vides détectées"

    sub.to_csv(output_path, index=False, encoding='utf-8')
    print(f'✅ Soumission sauvegardée : {output_path} ({len(sub)} lignes)')
    return sub

submission = make_submission(
    test['ID'], test_predictions,
    f'{SUBMISSIONS_DIR}/submission_gemma4_e2b_finetuned.csv',
    sample_submission['ID'],
)
submission.head(5)

## 7 — Aperçu par langue

In [ ]:
preview = test[['ID', 'subset', 'input']].copy()
preview['answer'] = submission['TargetRLF1']

for lang in sorted(preview['subset'].unique()):
    row = preview[preview['subset'] == lang].iloc[0]
    print(f"[{lang}] {row['input'][:90]}")
    print(f"  → {row['answer'][:150]}")
    print()

---
**Soumission prête** : `submissions/submission_gemma4_e2b_finetuned.csv` sur Drive. Téléchargez ce fichier et déposez-le sur la page du challenge Zindi pour obtenir le score officiel (`TargetRLF1`, `TargetR1F1`, `TargetLLM`).